## Local Physical and Chemical Consistency

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pymatgen.core import Structure

In [2]:
# Load supercell
s_super = Structure.from_file(
    filename = "../data/supercell.cif",
    site_tolerance = 0,
    frac_tolerance = 0,
)

# Load subcell
s_sub = Structure.from_file(
    filename = "../data/cut_subcell.cif",
    site_tolerance = 0,
    frac_tolerance = 0,
)

In [3]:
print(f"Composition of supercell: {s_super.composition}")
print(f"Composition of subcell: {s_sub.composition}")

print(f"Avg volume per atom of supercell: {s_super.volume/s_super.num_sites}")
print(f"Avg volume per atom of subcell: {s_sub.volume/s_sub.num_sites}")

print(f"Oxi state of supercell: {s_super.composition.oxi_state_guesses()}")
print(f"Oxi state of subcell: {s_sub.composition.oxi_state_guesses()}")

Composition of supercell: Li30 Ta30 Cl180
Composition of subcell: Li10 Ta7 Cl56
Avg volume per atom of supercell: 24.000000577471564
Avg volume per atom of subcell: 23.671232876712327
Oxi state of supercell: ({'Li': 1.0, 'Ta': 5.0, 'Cl': -1.0},)
Oxi state of subcell: []


### Physical Consistency  (Density)
- **Match the density of subcell to supercell**
- Match the density of local core shell to surrounding space

In [4]:
avg_vol_super = s_super.volume/s_super.num_sites
num_sub_ground = s_sub.volume/avg_vol_super
num_sub_add = num_sub_ground - s_sub.num_sites
num_sub_add = int(num_sub_add)

print(f"{num_sub_add} atoms need to be added to the subcell to match the volume of the supercell")

-1 atoms need to be added to the subcell to match the volume of the supercell


### Chemical Consistency (Charge Balance)

In [5]:
oxi_state_super = s_super.composition.oxi_state_guesses()[0]
oxi_state_super

{'Li': 1.0, 'Ta': 5.0, 'Cl': -1.0}

In [6]:
charge_sub_ground = 0
s_sub.add_oxidation_state_by_element(oxidation_states = oxi_state_super)
charge_sub_add = charge_sub_ground - s_sub.charge
charge_sub_add = int(charge_sub_add)
print(f"{charge_sub_add} positive charge need to be added to the subcell to match charge balance")

11 positive charge need to be added to the subcell to match charge balance


### Overall Balance

In [7]:
x + 5*y - z = 11
min(np.abs(x) + np.abs(y) + np.abs(z))

SyntaxError: cannot assign to operator (3058019397.py, line 1)

In [9]:
%%timeit
import pulp

# Define the problem
prob = pulp.LpProblem("IntegerOptimization", pulp.LpMinimize)

# Variables
x = pulp.LpVariable('x', cat='Integer')
y = pulp.LpVariable('y', cat='Integer')
z = pulp.LpVariable('z', cat='Integer')
abs_x = pulp.LpVariable('abs_x', lowBound=0)
abs_y = pulp.LpVariable('abs_y', lowBound=0)
abs_z = pulp.LpVariable('abs_z', lowBound=0)

# Objective function
prob += abs_x + abs_y + abs_z  # Minimize abs_x + abs_y + abs_z

# Constraints
prob += x + 5*y - z == 11
prob += x <= abs_x  # abs_x >= x
prob += -x <= abs_x  # abs_x >= -x
prob += y <= abs_y  # abs_y >= y
prob += -y <= abs_y  # abs_y >= -y
prob += z <= abs_z  # abs_z >= z
prob += -z <= abs_z  # abs_z >= -z

# Solve the problem
status = prob.solve()

# Print the result
print(pulp.value(x), pulp.value(y), pulp.value(z))

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/59962cc7522c42e2b28913b027e6358d-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/59962cc7522c42e2b28913b027e6358d-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 12 COLUMNS
At line 37 RHS
At line 45 BOUNDS
At line 49 ENDATA
Problem MODEL has 7 rows, 6 columns and 15 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 2.2 - 0.00 seconds
Cgl0003I 0 fixed, 4 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 6 rows, 5 columns (2 integer (0 of which binary)) and 14 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of 3 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0001I Search completed

In [10]:
%%timeit
import cvxpy as cp

# Variables
x = cp.Variable(integer=True)
y = cp.Variable(integer=True)
z = cp.Variable(integer=True)

# Objective function
objective = cp.Minimize(cp.square(x) + cp.square(y) + cp.square(z))

# Constraints
constraints = [x + 5*y - z == 11]

# Define the problem
prob = cp.Problem(objective, constraints)

# Solve the problem using ECOS_BB solver
prob.solve(solver=cp.ECOS_BB)

# Print the result
x, y, z = map(np.round, [x.value, y.value, z.value])
print(f"x: {x}, y: {y}, z: {z}")

x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, 

x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, z: -0.0
x: 1.0, y: 2.0, 